# Sber sbert_large_mt_nlu_ru — test vs rubert-tiny2

GPU accelerator ON. Encodes the 38 360-word ru vocab, computes top-30 and synonym ranks for 15 secrets: 10 common + 5 real rare round secrets. Output: `sbert_results.json` (download it and send it back). No query:/passage: prefixes — classic SBERT, bare words.

In [ ]:
!pip install -q sentence-transformers
import json, time, numpy as np, urllib.request
from sentence_transformers import SentenceTransformer

In [ ]:
# vocab: same file the game uses, from the public repo
url = 'https://raw.githubusercontent.com/alex-mtl/mao-dao/main/stream-engine/word-data/built/ru/vocab.txt'
vocab = [w.strip() for w in urllib.request.urlopen(url).read().decode('utf-8').splitlines() if w.strip()]
print('vocab', len(vocab))

In [ ]:
SECRETS = {
    'собака': ['пес', 'щенок', 'дворняга', 'овчарка'],
    'море': ['океан', 'залив', 'лагуна'],
    'машина': ['автомобиль', 'авто', 'тачка'],
    'дом': ['жилище', 'изба', 'коттедж', 'особняк'],
    'книга': ['роман', 'повесть', 'том', 'учебник'],
    'солнце': ['светило', 'заря'],
    'хлеб': ['булка', 'батон', 'каравай'],
    'друг': ['товарищ', 'приятель', 'дружок'],
    'работа': ['труд', 'дело', 'служба'],
    'город': ['мегаполис', 'поселение', 'столица'],
    'праотец': ['предок', 'отец', 'дед', 'прадед'],
    'пепино': ['огурец', 'дыня', 'плод'],
    'плотничество': ['плотник', 'столяр', 'ремесло'],
    'радиооператор': ['радист', 'радио', 'оператор'],
    'донос': ['жалоба', 'клевета', 'ябеда'],
}

In [ ]:
model = SentenceTransformer('ai-forever/sbert_large_mt_nlu_ru')
print('dim', model.get_sentence_embedding_dimension())
t0 = time.time()
emb = model.encode(vocab, batch_size=256, normalize_embeddings=True, show_progress_bar=True)
print(f'vocab encoded in {time.time()-t0:.0f}s shape {emb.shape}')
emb = np.asarray(emb, dtype=np.float32)
idx = {w: i for i, w in enumerate(vocab)}
out = {}
for secret, syns in SECRETS.items():
    sims = emb @ emb[idx[secret]]
    order = np.argsort(-sims, kind='stable')
    top = [(vocab[int(i)], int(r+1)) for r, i in enumerate(order[:30])]
    syn = {s: int(np.where(order == idx[s])[0][0]) + 1 for s in syns if s in idx}
    out[secret] = {'after_top30': top, 'after_syn_ranks': syn}
    print(f"AFTER {secret}: top={[w for w, _ in top[:6]]} syn={syn}", flush=True)
json.dump({'secrets': out}, open('sbert_results.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('wrote sbert_results.json — download this file from the notebook output')